# Retrieval contracts
Source-backed evidence, model/input identity, and index manifests shared by object and layout consumers.

In [ ]:
# | default_exp retrieval.contracts


In [ ]:
# | export
"""Shared retrieval contracts; source-backed evidence and explicit encoder identity."""

from dataclasses import asdict, dataclass, field
import hashlib
import json
import os
from pathlib import Path
import re
import tempfile
from typing import Callable


def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False)


def fingerprint(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', dir=path.parent, delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(canonical_json(value))
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def token_count(text, tokenizer=None):
    """Use the supplied model tokenizer or a conservative UTF-8 byte upper bound."""
    return len(tokenizer(text)) if tokenizer is not None else len(text.encode('utf-8'))


@dataclass(frozen=True)
class EncoderSpec:
    model: str
    model_digest: str
    dimension: int
    normalization: str = 'provider'
    provider_options: dict = field(default_factory=dict)
    query_template: str = '{text}'
    document_template: str = '{text}'
    max_input_tokens: int = 8192
    tokenizer: str = 'utf8-upper-bound'
    overflow_policy: str = 'reject'

    def __post_init__(self):
        if not self.model or not self.model_digest or isinstance(self.dimension, bool) or self.dimension < 1:
            raise ValueError('An encoder requires model, resolved digest and positive dimension')
        if self.normalization not in {'provider', 'l2'}: raise ValueError('Unsupported normalization policy')
        if self.max_input_tokens < 1 or self.overflow_policy not in {'reject', 'split'}:
            raise ValueError('Invalid encoder input budget or overflow policy')
        for template in (self.query_template, self.document_template):
            if template.count('{text}') != 1:
                raise ValueError('Encoder templates must contain exactly one {text} placeholder')
        if self.tokenizer != 'utf8-upper-bound':
            raise ValueError('Register a verified tokenizer before using a non-default tokenizer identity')

    @property
    def identity(self):
        return fingerprint(asdict(self))

    def render(self, text, *, query=False):
        rendered = (self.query_template if query else self.document_template).replace('{text}', text)
        if token_count(rendered) > self.max_input_tokens:
            raise ValueError('Encoder input exceeds its token budget; split the source view before embedding')
        return rendered

    def request(self, text, *, query=False):
        return dict(model=self.model, input=self.render(text, query=query), truncate=False,
                    options=dict(self.provider_options))


@dataclass(frozen=True)
class RepresentationSpec:
    kind: str = 'summary'
    version: int = 1
    context: bool = True

    def __post_init__(self):
        if self.kind not in {'summary', 'source'} or self.version < 1:
            raise ValueError('Unknown representation kind or version')

    def render(self, record):
        text = record.summary if self.kind == 'summary' else record.content
        if self.kind == 'source' and self.context:
            attrs = record.attributes
            context = [*record.section_headings, str(attrs.get('caption', ''))]
            if attrs.get('headers'):
                context.append(canonical_json(attrs['headers']))
            text = '\n'.join(value for value in [*context, text] if value)
        return text


@dataclass
class EvidenceItem:
    id: str
    generation: str
    node_ids: list[str]
    source_file: str
    source_hash: str
    revision: dict
    spans: list[dict]
    text: str
    retrieval_routes: list[str] = field(default_factory=list)
    expansion: list[dict] = field(default_factory=list)
    attributions: list[dict] = field(default_factory=list)

    def validate(self):
        if not re.fullmatch(r'E[1-9][0-9]*', self.id) or not self.text.strip():
            raise ValueError('Evidence must have an issued ID and nonempty source text')
        if not self.node_ids or not self.generation or not self.source_file or not self.source_hash:
            raise ValueError('Evidence requires source occurrence, generation, file and hash')
        if not isinstance(self.revision, dict) or not self.spans:
            raise ValueError('Evidence requires revision metadata and source locators')


class EvidenceRegistry:
    version = 1

    def __init__(self, items=()):
        self.items = []
        self._identities = {}
        for value in items:
            item = value if isinstance(value, EvidenceItem) else EvidenceItem(**value)
            item.validate()
            if item.id != f'E{len(self.items)+1}':
                raise ValueError('Evidence registry IDs must be uniquely issued in order')
            self.items.append(item)
            self._identities[fingerprint([item.generation, item.source_file, item.spans, item.source_hash, item.revision, item.text])] = item

    @classmethod
    def from_result(cls, result):
        if result.get('evidence_schema') != cls.version or 'evidence' not in result:
            raise ValueError('Saved retrieval lacks a structured evidence registry; retrieve again')
        return cls(result['evidence'])

    def issue(self, *, record, generation, manual, text, routes=(), expansion=()):
        if not text.strip():
            return None
        attrs = record.get('attributes', {})
        spans = attrs.get('source_spans') or [dict(node_id=record['id'],
            line_start=attrs.get('line_start'), line_end=attrs.get('line_end'))]
        spans = list(spans) + [dict(value) for value in expansion if value.get('node_id') and value.get('line_start') is not None]
        revision = dict(manual)
        for key in ('source_revision_id', 'logical_document_id', 'source_identity_status'):
            if key in attrs: revision[key] = attrs[key]
        source_hash = attrs.get('source_hash') or hashlib.sha256(record.get('content', '').encode()).hexdigest()
        attribution = dict(node_id=record['id'], source_file=record.get('source_file', ''), revision=revision)
        identity = fingerprint([generation, record.get('source_file') or record.get('semantics_file') or 'unknown-source', spans, source_hash, revision, text])
        if identity in self._identities:
            item = self._identities[identity]
            if record['id'] not in item.node_ids:
                item.node_ids.append(record['id'])
                item.attributions.append(attribution)
            return item
        item = EvidenceItem(f'E{len(self.items)+1}', generation, [record['id']],
            record.get('source_file') or record.get('semantics_file') or 'unknown-source',
            source_hash, revision, list(spans), text, list(routes), list(expansion), [attribution])
        item.validate()
        self.items.append(item)
        self._identities[identity] = item
        return item

    @staticmethod
    def render_item(item):
        first = item.spans[0]
        return (f'[{item.id}] {item.revision} | {item.source_file} | '
                f'lines {first.get("line_start")}-{first.get("line_end")} | {", ".join(item.node_ids)}\n{item.text}')

    def render(self):
        return '\n\n'.join(self.render_item(item) for item in self.items)

    def payload(self):
        return dict(evidence_schema=self.version, evidence=[asdict(item) for item in self.items])

    def cited(self, labels, *, comparison=False):
        by_id = {item.id: item for item in self.items}
        if not isinstance(labels, list) or not labels or any(not isinstance(label, str) or label not in by_id for label in labels):
            raise ValueError('Cite issued evidence registry labels')
        items = [by_id[label] for label in dict.fromkeys(labels)]
        immutable = {}
        for item in items:
            document = item.revision.get('logical_document_id') or item.revision.get('key') or item.source_file
            source_revision = item.revision.get('source_revision_id')
            if source_revision: immutable.setdefault(document, set()).add(source_revision)
        if not comparison and any(len(revisions) > 1 for revisions in immutable.values()):
            raise ValueError('Citations mix incompatible immutable source revisions')
        versions = {(item.revision.get('profile', ''), item.revision.get('revision', '')) for item in items}
        if not comparison and len(versions) > 1:
            raise ValueError('Citations mix incompatible corpus/manual revisions')
        return items

    def manual_label(self, labels, *, comparison=False):
        items = self.cited(labels, comparison=comparison)
        return '; '.join(dict.fromkeys(
            f"{item.revision.get('key') or Path(item.source_file).stem}, corpus {item.revision.get('profile') or 'unspecified'}, revision {item.revision.get('revision') or 'unknown'}"
            for item in items))


@dataclass
class IndexManifest:
    artifact_kind: str
    collection: str
    catalog_id: str
    encoder_fingerprint: str
    supported_kinds: list[str]
    representations: list[str]
    active_generations: dict[str, str]
    source_identity: str = 'legacy-path-aliases'
    embedding_config: dict = field(default_factory=dict)
    encoder_spec: dict | None = None
    version: int = 1

    def validate(self):
        if self.version != 1 or self.artifact_kind not in {'semantics', 'tasks', 'mixed'}:
            raise ValueError('Unsupported index manifest artifact kind/version')
        if not self.collection or not self.catalog_id or not self.encoder_fingerprint:
            raise ValueError('Incomplete index manifest identity')
        if not self.representations or not set(self.representations) <= {'summary', 'source'}:
            raise ValueError('Unsupported index representation')

    def write(self, directory):
        self.validate()
        atomic_json(Path(directory)/'index-manifest.json', asdict(self))

    @classmethod
    def read(cls, directory, *, required=False):
        path = Path(directory)/'index-manifest.json'
        if not path.exists():
            if required:
                raise ValueError(f'Index manifest missing: {path}')
            return None
        result = cls(**json.loads(path.read_text()))
        result.validate()
        return result


def default_index_directory(root, model, *, use_tasks=True):
    return Path(root)/'db'/('task_embeddings' if use_tasks else 'object_embeddings')/model.replace('/', '_').replace(':', '-')


def resolve_index_directory(root, model, *, override=None, use_tasks=True):
    path = Path(override).expanduser() if override else default_index_directory(root, model, use_tasks=use_tasks)
    return path if path.is_absolute() else Path(root)/path


def lexical_terms(text):
    """Exact identifiers plus Latin terms and CJK characters/bigrams."""
    words = re.findall(r'[A-Za-z0-9_]+(?:[./:-][A-Za-z0-9_]+)*', text.casefold())
    for run in re.findall(r'[\u3400-\u9fff]+', text):
        words.extend(run)
        words.extend(run[i:i+2] for i in range(len(run)-1))
    return list(dict.fromkeys(words))


def reciprocal_rank_fusion(channels, *, constant=60, weights=None):
    scores, provenance = {}, {}
    for name, candidates in channels.items():
        seen = set()
        for rank, key in enumerate(candidates, 1):
            if key in seen:
                continue
            seen.add(key)
            scores[key] = scores.get(key, 0.0) + (weights or {}).get(name, 1.0)/(constant + rank)
            provenance.setdefault(key, []).append(name)
    return [(key, scores[key], provenance[key]) for key in sorted(scores, key=lambda k: (-scores[k], str(k)))]

from contextlib import contextmanager
import time
from uuid import uuid4


@contextmanager
def index_lock(directory):
    """Serialize snapshot pins, publication and cleanup across local processes."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / '.publication.lock').open('a+b') as stream:
        if os.name == 'nt':
            import msvcrt
            stream.seek(0); stream.write(b'0'); stream.flush(); stream.seek(0)
            msvcrt.locking(stream.fileno(), msvcrt.LK_LOCK, 1)
        else:
            import fcntl
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX)
        try:
            yield
        finally:
            if os.name == 'nt':
                stream.seek(0); msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


def pin_generations(directory, generations):
    """Create under index_lock. A crashed reader's pin conservatively prevents cleanup."""
    path = Path(directory)/'reader-pins'/(uuid4().hex+'.json')
    atomic_json(path, dict(generations=sorted(set(generations)), process=os.getpid(), created=time.time()))
    return path


def protected_generations(directory):
    protected = set()
    for folder, key in [('reader-pins', 'generations'), ('preparations', 'generations')]:
        for path in (Path(directory)/folder).glob('*.json'):
            # Unreadable pins fail closed instead of allowing deletion.
            payload = json.loads(path.read_text())
            protected.update(payload[key])
    return protected

async def verify_encoder(client, spec):
    """Verify pooled/Ollama metadata without inference; custom providers report digest per response."""
    if hasattr(client, 'check_models'):
        await client.check_models(spec.model, expected_digest=spec.model_digest, require_digest=True)
        return True
    if hasattr(client, 'list'):
        response = await client.list()
        models = response.get('models', []) if isinstance(response, dict) else getattr(response, 'models', [])
        for item in models:
            name = item.get('model', item.get('name')) if isinstance(item, dict) else (getattr(item, 'model', None) or getattr(item, 'name', None))
            if name in {spec.model, spec.model+':latest'}:
                digest = item.get('digest') if isinstance(item, dict) else getattr(item, 'digest', None)
                if digest != spec.model_digest: raise ValueError('Resolved provider model digest does not match EncoderSpec')
                return True
        raise ValueError('Configured encoder model not found in provider metadata')
    return False
